# 02 — Train Tier 1: QLoRA SFT

**Runs on:** Google Colab Free (T4 15 GB).

## How to use this notebook
1. Open in Colab. Set runtime → GPU (T4).
2. (Optional) Edit the `MODEL_NAME` in the **CONFIG** cell to the winner from `02a_model_evaluation.ipynb`. Default is Phi-4-mini.
3. **Runtime → Run all.**
4. When prompted, upload `train.jsonl`, `val.jsonl`, `test.jsonl` (from `tools/yen_sei/data/refined/`).
5. Wait ~3–6 hours. The notebook will:
   - QLoRA fine-tune the chosen model (3 epochs, early stop on val loss).
   - Run inference on the test split + report JSON-compliance and quality metrics.
   - Save adapter, merged-fp16 weights (if VRAM allows), and zip them for download.

**Total user effort: pick the runtime, click Run all, drop three files. No copy-paste of code.**


In [ ]:
# =====================================================================
# Cell 1 — Install dependencies (silent). ~2-3 min on first run.
# =====================================================================
import subprocess, sys, os, time

PKGS = [
    "unsloth",
    "peft>=0.11.0",
    "transformers>=4.45.0",
    "datasets>=2.20.0",
    "bitsandbytes>=0.43.0",
    "accelerate>=0.34.0",
    "trl>=0.10.0",
]
print("Installing dependencies (silent)…")
t0 = time.time()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *PKGS],
    check=True,
)
print(f"Done in {time.time() - t0:.1f}s.")


In [ ]:
# =====================================================================
# Cell 2 — CONFIG. Edit MODEL_NAME if 02a picked Gemma instead of Phi.
# =====================================================================

# Default winner; override after running 02a.
MODEL_NAME  = "microsoft/Phi-4-mini-instruct"

# Hyperparameters from PLAN.md
MAX_SEQ_LEN  = 2048
EPOCHS       = 3
BATCH_SIZE   = 4
GRAD_ACCUM   = 4         # effective batch = 16
LR           = 2e-4
LORA_R       = 32
LORA_ALPHA   = 64
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

# Limits (set to None to use all). Useful for quick smoke runs.
TRAIN_LIMIT  = None      # full ~2,859 train rows
VAL_LIMIT    = None      # full ~227 val rows
TEST_LIMIT   = None      # full ~231 test rows

# Output / save behaviour
OUT_DIR             = "/content/yen_sei_tier1"
SAVE_MERGED_FP16    = True   # merge LoRA into base + save fp16 (large; download as zip)
PUSH_TO_HF          = False  # set True + add HF_TOKEN below to push adapter to your hub
HF_REPO_ID          = ""     # e.g. "your-username/yen-sei-tier1-phi4-lora"
HF_TOKEN            = ""     # paste a write token if PUSH_TO_HF=True

os.makedirs(OUT_DIR, exist_ok=True)
print(f"MODEL_NAME = {MODEL_NAME}")
print(f"OUT_DIR    = {OUT_DIR}")


## Upload data

The next cell looks for `train.jsonl`, `val.jsonl`, `test.jsonl` in `/content/`. If missing, it opens an upload widget — drop all three files from `tools/yen_sei/data/refined/`.


In [ ]:
# =====================================================================
# Cell 3 — Locate (or upload) train/val/test JSONL files.
# =====================================================================
import json
from pathlib import Path

REQUIRED = ["train.jsonl", "val.jsonl", "test.jsonl"]
SEARCH_DIRS = [Path("/content"), Path.cwd(), Path("/content/drive/MyDrive/yen_sei")]

def find_data():
    found = {}
    for name in REQUIRED:
        for d in SEARCH_DIRS:
            p = d / name
            if p.exists():
                found[name] = p
                break
    return found

found = find_data()
missing = [n for n in REQUIRED if n not in found]

if missing:
    print(f"Missing: {missing}. Opening upload widget…")
    try:
        from google.colab import files  # type: ignore
        uploaded = files.upload()
        for name in uploaded:
            dest = Path("/content") / name
            print(f"  saved {name} -> {dest} ({len(uploaded[name])} bytes)")
        found = find_data()
    except ImportError:
        raise SystemExit(
            "Not in Colab. Place train.jsonl, val.jsonl, test.jsonl next to this "
            "notebook or in /content/."
        )

assert all(n in found for n in REQUIRED), f"Still missing: {set(REQUIRED) - set(found)}"
TRAIN_PATH = found["train.jsonl"]
VAL_PATH   = found["val.jsonl"]
TEST_PATH  = found["test.jsonl"]
print(f"\ntrain: {TRAIN_PATH}\nval:   {VAL_PATH}\ntest:  {TEST_PATH}")

def load_jsonl(path, limit=None):
    rows = []
    with open(path, "r", encoding="utf-8-sig") as f:
        for line in f:
            rows.append(json.loads(line))
            if limit and len(rows) >= limit:
                break
    return rows

train_rows = load_jsonl(TRAIN_PATH, limit=TRAIN_LIMIT)
val_rows   = load_jsonl(VAL_PATH,   limit=VAL_LIMIT)
test_rows  = load_jsonl(TEST_PATH,  limit=TEST_LIMIT)
print(f"\nLoaded train={len(train_rows)}, val={len(val_rows)}, test={len(test_rows)}.")


## Train (QLoRA SFT)

Loads the model in 4-bit, attaches LoRA adapters, runs SFT for `EPOCHS` epochs with cosine LR schedule and warmup. Saves the best checkpoint by val loss. ~3-6 hours on T4 for the full dataset.


In [ ]:
# =====================================================================
# Cell 4 — Load model + LoRA adapter, prepare dataset, train.
# =====================================================================
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

print(f"Loading {MODEL_NAME} in 4-bit…")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules="all-linear",
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

def to_text(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )}

train_ds = Dataset.from_list(train_rows).map(to_text, remove_columns=["messages"])
val_ds   = Dataset.from_list(val_rows).map(to_text,   remove_columns=["messages"])

steps_per_epoch = max(1, len(train_ds) // (BATCH_SIZE * GRAD_ACCUM))
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = max(1, int(total_steps * WARMUP_RATIO))
print(f"\nTotal steps: {total_steps}  (steps/epoch={steps_per_epoch}, warmup={warmup_steps})")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=SFTConfig(
        output_dir=os.path.join(OUT_DIR, "checkpoints"),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        weight_decay=WEIGHT_DECAY,
        logging_steps=20,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        report_to="none",
        seed=42,
    ),
)

t0 = time.time()
trainer.train()
train_secs = time.time() - t0
peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0
print(f"\nTraining done in {train_secs/60:.1f} min, peak VRAM: {peak_vram:.1f} GB")


## Evaluate on test set
